In [1]:
import sys, joblib, pandas as pd
from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field, ValidationError

sys.path.append("Modelos")
pipe = joblib.load("Modelos/modelo_tabular.joblib")   # Pipeline GaussianNB, classes [0, 1]

SimNao = Literal["Yes", "No"]
ServInternet = Literal["Yes", "No", "No internet service"]

class ClienteTelco(BaseModel):   # mesma ordem das colunas do treino
    gender: Literal["Female", "Male"]
    SeniorCitizen: Literal[0, 1]
    Partner: SimNao
    Dependents: SimNao
    tenure: int = Field(ge=0, le=72)
    PhoneService: SimNao
    MultipleLines: Literal["Yes", "No", "No phone service"]
    InternetService: Literal["DSL", "Fiber optic", "No"]
    OnlineSecurity: ServInternet
    OnlineBackup: ServInternet
    DeviceProtection: ServInternet
    TechSupport: ServInternet
    StreamingTV: ServInternet
    StreamingMovies: ServInternet
    Contract: Literal["Month-to-month", "One year", "Two year"]
    PaperlessBilling: SimNao
    PaymentMethod: Literal["Bank transfer (automatic)", "Credit card (automatic)",
                           "Electronic check", "Mailed check"]
    MonthlyCharges: float = Field(ge=0)
    TotalCharges: float = Field(ge=0)

ROTULOS = {0: "Não cancela", 1: "Cancela (churn)"}

def prever_tabular(dados: dict) -> dict:
    x = ClienteTelco(**dados)
    p = pipe.predict_proba(pd.DataFrame([x.model_dump()]))[0]
    i = int(p.argmax())
    return {"classe": ROTULOS[int(pipe.classes_[i])], "prob": round(float(p[i]), 4)}

cliente = {"gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
           "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service",
           "InternetService": "DSL", "OnlineSecurity": "No", "OnlineBackup": "Yes",
           "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
           "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
           "PaymentMethod": "Electronic check", "MonthlyCharges": 29.85, "TotalCharges": 29.85}
print(prever_tabular(cliente))   # esperado: {'classe': 'Cancela (churn)', 'prob': 0.9999}

{'classe': 'Cancela (churn)', 'prob': 0.9999}
